# Step 06: Chat with Multiple Plugins

This notebook demonstrates interactive chat using multiple plugins (Location and DateTime).

In [14]:
import asyncio
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv, find_dotenv
dotenv_path = find_dotenv()
load_dotenv(dotenv_path, override=True)

True

In [15]:
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import (
    AzureChatCompletion,
    AzureChatPromptExecutionSettings,
)
from semantic_kernel.contents.chat_history import ChatHistory
from plugins.datetime import DateTimePlugin
from plugins.location import LocationPlugin

In [16]:
# Initialize kernel and plugins
kernel = Kernel()
azure_chat_completion = AzureChatCompletion()
kernel.add_service(azure_chat_completion)

kernel.add_plugin(LocationPlugin(), plugin_name="location_plugin")
kernel.add_plugin(DateTimePlugin(), plugin_name="time_plugin")

execution_settings = AzureChatPromptExecutionSettings()

In [17]:
# Get location and time context
location_function = kernel.get_function(
    plugin_name="location_plugin", function_name="GetLocationInfo"
)
location_info = await kernel.invoke(location_function)

time = kernel.get_function(plugin_name="time_plugin", function_name="Time")
current_time = await kernel.invoke(time)

print(f"Location: {location_info}")
print(f"Time: {current_time}")

Location: Athlone, Leinster, IE
Coordinates: 53.4228,-7.9372
Timezone: Europe/Dublin
Time: 03:28:56 PM


In [18]:
# Initialize chat history with context
history = ChatHistory()
history.add_system_message("User location: " + str(location_info))
history.add_system_message("Time that User started this chat: " + str(current_time))

In [19]:
# Interactive chat function
async def chat(user_input: str):
    history.add_user_message(user_input)
    
    result = await azure_chat_completion.get_chat_message_content(
        chat_history=history,
        settings=execution_settings,
        kernel=kernel,
    )
    
    print("Assistant > " + str(result))
    history.add_message(result)
    return result

In [20]:
# Example chat interaction
await chat("What's the weather like where I am?")

Assistant > I currently can't access real-time weather data. However, you can check the weather in Athlone using trusted sources like [Met Éireann](https://www.met.ie) or a weather app that provides updates for your area.


ChatMessageContent(inner_content=ChatCompletion(id='chatcmpl-Ds81hzioAjMTqxtNQ3g3mCwev2DHT', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="I currently can't access real-time weather data. However, you can check the weather in Athlone using trusted sources like [Met Éireann](https://www.met.ie) or a weather app that provides updates for your area.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'protected_material_code': {'detected': False, 'filtered': False}, 'protected_material_text': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1781792937, model='gpt-4o-2024-11-20', object='chat.completion', moderation=None, service_tier='default', system_fingerp